In [1]:
import numpy as np
import pandas as pd
from margins import margins

pd.options.display.float_format = '{:.3f}'.format
MM2IN = 0.0393701


# CHAMBER (304 Stainless Steel)

In [2]:
#From Nickel Institute: 
sigma_temp = 9900

# approximate as thick walled cylinder
p = np.array([300, 500]) #pressure [psi]

r_i = 0.377 # radius [in]
r_o = 1.654/2 # outer radius [in] (approximate as the outer radius of a cylinder)


# Compute von mises stress
sigma_hoop = p*(r_o**2 + r_i**2) / (r_o**2 - r_i**2)
sigma_axial = p * r_i**2 / (r_o**2 - r_i**2)
sigma = ((sigma_hoop**2 + sigma_axial**2 + p**2 - sigma_hoop*sigma_axial + p*sigma_hoop + p*sigma_axial))**0.5

SF = 2.5
FOS_actual =  sigma_temp / sigma
margin = margins(SF, sigma_temp, sigma)
df_Stress = pd.DataFrame({
    "Chamber Pressure[psi]":p,
    "Design Stress [psi]": sigma,
    "Yield Margin [%]": margin,
    "Actual FoS": FOS_actual
})

print("Design FoS:", SF)
print(f"Wall Thickness {r_o-r_i:0.3f} in")
df_Stress


Design FoS: 2.5
Wall Thickness 0.450 in


,Chamber Pressure[psi],Design Stress [psi],Yield Margin [%],Actual FoS
0,300,655.925,503.728,15.093
1,500,1093.208,262.237,9.056


In [9]:
#From Nickel Institute: 
sigma_temp = 9900

# approximate as thick walled cylinder
p = np.array([300, 500]) #pressure [psi]

r_i = 0.148 # radius [in]
r_o = 0.295 # outer radius [in] (approximate as the outer radius of a cylinder)


# Compute von mises stress
sigma_hoop = p*(r_o**2 + r_i**2) / (r_o**2 - r_i**2)
sigma_axial = p * r_i**2 / (r_o**2 - r_i**2)
sigma = ((sigma_hoop**2 + sigma_axial**2 + p**2 - sigma_hoop*sigma_axial + p*sigma_hoop + p*sigma_axial))**0.5

SF = 2.5
FOS_actual =  sigma_temp / sigma
margin = margins(SF, sigma_temp, sigma)
df_Stress = pd.DataFrame({
    "Chamber Pressure[psi]":p,
    "Design Stress [psi]": sigma,
    "Yield Margin [%]": margin,
    "Actual FoS": FOS_actual
})

print("Design FoS:", SF)
print(f"Wall Thickness {r_o-r_i:0.3f} in")
df_Stress


Design FoS: 2.5
Wall Thickness 0.147 in


,Chamber Pressure[psi],Design Stress [psi],Yield Margin [%],Actual FoS
0,300,694.392,470.283,14.257
1,500,1157.320,242.170,8.554


# Flame Tube (Copper 715)

In [3]:
#Nickel Institue
sigma_temp = 20000 #[psi]

# thick walled cylinder
p = np.array([300,500]) #pressure [psi]

r_o = 0.477 # actual thickness [in] 
r_i = 0.148 # radius [in]


# Compute von mises stress
sigma_hoop = p*(r_o**2 + r_i**2) / (r_o**2 - r_i**2)
sigma_axial = p * r_i**2 / (r_o**2 - r_i**2)
sigma = ((sigma_hoop**2 + sigma_axial**2 + p**2 - sigma_hoop*sigma_axial + p*sigma_hoop + p*sigma_axial))**0.5

SF = 2.5
FOS_actual =  sigma_temp / sigma
margin = margins(SF, sigma_temp, sigma)
df_Stress = pd.DataFrame({
    "Chamber Pressure[psi]":p,
    "Design Stress [psi]": sigma,
    "Yield Margin [%]": margin,
    "Actual FoS": FOS_actual
})

print("Design FoS:", SF)
print(f"Wall Thickness {r_o-r_i:0.3f} in")
df_Stress


Design FoS: 2.5
Wall Thickness 0.329 in


,Chamber Pressure[psi],Design Stress [psi],Yield Margin [%],Actual FoS
0,300,574.967,1291.385,34.785
1,500,958.278,734.831,20.871


In [4]:
P= 500 #psi
FS = 2.5
sigma_vm = sigma_temp / FS
k = np.sqrt(sigma_vm / (sigma_vm - np.sqrt(3) * P))

In [5]:
t = r_i*(k-1)
t

np.float64(0.00872595672308426)

In [6]:
import pandas as pd

# Inputs
sigma_temp = 9900  # [psi]
SF = 2.5
target_stress = sigma_temp / SF

p = np.array([300, 500]) # [psi]
r_i = 0.377 # [in]

# Solving for k = r_o / r_i using the Von Mises distortion energy theory
# For internal pressure only, the ratio k is:
# k = sqrt( (sigma_v + P*sqrt(3)) / (sigma_v - P*sqrt(3)) ) --- This is an approximation.
# For exactness, we use the quadratic form of the von mises equation:
def calculate_required_thickness(P, r_i, target_sigma):
    # k = r_o / r_i. Solving the Von Mises eq for k^2:
    # sigma_v = (sqrt(3) * P * k^2) / (k^2 - 1)
    # k^2 = sigma_v / (sigma_v - sqrt(3) * P)
    
    k_sq = target_sigma / (target_sigma - np.sqrt(3) * P)
    k = np.sqrt(k_sq)
    r_o_req = k * r_i
    return r_o_req - r_i

t_req = calculate_required_thickness(p, r_i, target_stress)

# Validation
r_o_new = r_i + t_req
sigma_h = p * (r_o_new**2 + r_i**2) / (r_o_new**2 - r_i**2)
sigma_a = p * (r_i**2) / (r_o_new**2 - r_i**2)
sigma_r = -p
sigma_v = (0.5 * ((sigma_h - sigma_a)**2 + (sigma_a - sigma_r)**2 + (sigma_r - sigma_h)**2))**0.5

df_results = pd.DataFrame({
    "Pressure [psi]": p,
    "Required Thickness [in]": t_req,
    "Resulting Stress [psi]": sigma_v,
    "Target Stress [psi]": target_stress
})

print(df_results)

   Pressure [psi]  Required Thickness [in]  Resulting Stress [psi]  \
0             300                    0.027                3960.000   
1             500                    0.050                3960.000   

   Target Stress [psi]  
0             3960.000  
1             3960.000  


In [7]:
#From Nickel Institute: 
sigma_temp = 9900

# approximate as thick walled cylinder
p = np.array([300, 500]) #pressure [psi]

r_i = 0.377 # radius [in]
r_o = r_i + 0.1 # outer radius [in] (approximate as the outer radius of a cylinder)


# Compute von mises stress
sigma_hoop = p*(r_o**2 + r_i**2) / (r_o**2 - r_i**2)
sigma_axial = p * r_i**2 / (r_o**2 - r_i**2)
sigma = ((sigma_hoop**2 + sigma_axial**2 + p**2 - sigma_hoop*sigma_axial + p*sigma_hoop + p*sigma_axial))**0.5

SF = 2.5
FOS_actual =  sigma_temp / sigma
margin = margins(SF, sigma_temp, sigma)
df_Stress = pd.DataFrame({
    "Chamber Pressure[psi]":p,
    "Design Stress [psi]": sigma,
    "Yield Margin [%]": margin,
    "Actual FoS": FOS_actual
})

print("Design FoS:", SF)
print(f"Wall Thickness {r_o-r_i:0.3f} in")
df_Stress


Design FoS: 2.5
Wall Thickness 0.100 in


,Chamber Pressure[psi],Design Stress [psi],Yield Margin [%],Actual FoS
0,300,1384.397,186.045,7.151
1,500,2307.329,71.627,4.291
